In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from src.utils.config_loader import load_config
from src.data.generate_sample import generate_sample
from src.preprocessing.data_preprocessing import preprocess_dataframe
from src.train.train_adaline import train_adaline
from src.predict.predict_adaline import predict_text

In [ ]:
config = load_config()
config.model_dump()

In [ ]:
df_sample = generate_sample()
df_sample.head()

In [ ]:
df = pd.read_csv(config.paths.preprocessed_data)
df.head()

In [ ]:
df['sentiment'].value_counts().plot(kind="bar", title="Class Distribution")
plt.show()

In [ ]:
df["len"] = df["text"].str.len()
df["len"].hist(bins=40)
plt.title("Distribution of Tweet Lengths")
plt.show()

In [ ]:
df.sample(5)[["text", "sentiment"]]

In [ ]:
result = train_adaline()

model = result["model"]
vectorizer = result["vectorizer"]
scaler = result["scaler"]
metrics = result["metrics"]

metrics["accuracy"]

In [ ]:
plt.plot(metrics["loss_history"])
plt.title("ADALINE Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.grid()
plt.show()

In [ ]:
print(classification_report(
    metrics["classification_report"]["0"]["support"] * [0] +
    metrics["classification_report"]["1"]["support"] * [1],
    [0] * metrics["classification_report"]["0"]["support"] +
    [1] * metrics["classification_report"]["1"]["support"],
))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

y_true = []
for _ in range(metrics["classification_report"]["0"]["support"]):
    y_true.append(0)
for _ in range(metrics["classification_report"]["1"]["support"]):
    y_true.append(1)

y_pred = model.predict(vectorizer.transform(df["text"]).toarray())

cm = confusion_matrix(y_true, y_pred[:len(y_true)])

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
predict_text("I love this project!", model, vectorizer, scaler)

In [ ]:
predict_text("This is awful and I hate it", model, vectorizer, scaler)

In [ ]:
texts = [
    "This is amazing!",
    "Terrible experience, hate everything.",
    "Not sure how I feel.",
    "The product is great!",
    "Worst service ever."
]

[predict_text(t, model, vectorizer, scaler) for t in texts]


In [ ]:
from src.utils.config_loader import load_config
config = load_config()
config.paths.model_dump() if hasattr(config.paths, "model_dump") else config.paths.__dict__
config.paths.raw_data